In [ ]:
import pandas as pd
import numpy as np

from skforecast.exceptions import IgnoredArgumentWarning
from skforecast.exceptions import LongTrainingWarning

import seaborn as sns
sns.set_theme(palette="colorblind")

import warnings
warnings.simplefilter('ignore', category=IgnoredArgumentWarning)
warnings.simplefilter('ignore', category=LongTrainingWarning)

# Ex.7.1.

The data for the exercise contains monthly expenditure on cafes, restaurants and takeaway food services in Victoria (Australia) from April 1982 up to April 2024. Perform the following steps:

1. Establish if the series contains any seasonality. If yes, determine the number of observations within one seasonal cycle.
2. Run backtesting of AutoArima, using three steps ahead forecasting and the initial training set size of 400.
3. Run backtesting of AutoETS and a Last-Value Baseline on the same data with the same evaluation settings.
4. Plot backtest predictions for AutoArima and AutoETS.
5. Print the evaluation metrics for the three methods.

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/expenditures_australia.csv",
                 parse_dates=["date"], index_col="date")
df.index.freq = 'MS'
df = df[df.index <= '2020-01-01']
df.info()

In [ ]:
# plot the series
???

In [ ]:
from skforecast.stats._arima import Arima 

from skforecast.model_selection import TimeSeriesFold
from skforecast.recursive import ForecasterStats
from skforecast.model_selection._validation import backtesting_stats
from sklearn.metrics import root_mean_squared_error


cv = TimeSeriesFold(
         steps=???, # =forecast horizon
         initial_train_size=???, # size of the initial training set
         refit=True, # if true, the forecaster is refitted at each fold
         fixed_train_size=True # fixed-size or expanding training data
     )

arima_forecaster = ForecasterStats(estimator=Arima(order=???, seasonal_order=???, m=???, stepwise=False))

results_arima, predictions_backtest = backtesting_stats(
                                   forecaster=???,
                                   y=???,
                                   cv=???,
                                   metric=[root_mean_squared_error, 
                                           'mean_absolute_error'],
                                   n_jobs='auto',
                                   verbose=False,
                                   show_progress=True,
                                   suppress_warnings=True
                               )
results_arima.index = ["ARIMA"]

In [ ]:
# plot predictions vs real values
tdf = pd.DataFrame({"y": ???, "y_hat": ???})
tdf.plot(figsize=(12,4))

In [ ]:
# compare with ETS

from skforecast.stats._ets import Ets

ets_forecaster = ForecasterStats(estimator=Ets(model=???))

results_ets, predictions_backtest = backtesting_stats(
                                   forecaster=???,
                                   y=???,
                                   cv=???,
                                   metric=[root_mean_squared_error, 
                                           'mean_absolute_error'],
                                   n_jobs='auto',
                                   verbose=False,
                                   show_progress=True
                               )
results_ets.index = ["ETS"]

In [ ]:
tdf = pd.DataFrame({"y": ???, "y_hat": ???})
tdf.plot(figsize=(12,4))

In [ ]:
# compare to last-value baseline

from skforecast.recursive import ForecasterEquivalentDate
from skforecast.model_selection._validation import backtesting_forecaster

last_value_forecaster = ForecasterEquivalentDate(offset=1, n_offsets=1)

results_lv, predictions_backtest = backtesting_forecaster(
                                   forecaster=???,
                                   y=???,
                                   cv=???,
                                   metric=[root_mean_squared_error, 
                                           'mean_absolute_error'],
                                   n_jobs='auto',
                                   verbose=False,
                                   show_progress=True
                               )
results_lv.index = ["Last-Value"]

In [ ]:
results_all = pd.concat([results_arima, results_ets, results_lv], axis=0)
results_all

# Ex 7.2

Use the lecture example with the bike sharing dataset, to train a Decision Tree model, instead of a Random Forest one. During HP tuning, use `max_depth` and `min_samples_split`, and specify 4 different settings for each. Answer the questions:

- At the start of HP tuning, how many observations were used for model training and how many for validation? How many were used for model evaluation on the test set?
- What are the HPs of the best DT model? What is the optimal number of lags for this model?
- Does the HP-tuned DT model improve on the last-value baseline? 
- How does it compare to the Random Forest model?

In [ ]:
# download and preprocess the data

df = pd.read_csv("https://raw.githubusercontent.com/skforecast/skforecast-datasets/main/data/bike_sharing_dataset_clean.csv", 
                 parse_dates=["date_time"], index_col="date_time")
df = df.asfreq('h')
df = df.drop(columns=["weather", "atemp"])

# train-test split

from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(df, test_size=0.2, random_state=7, shuffle=False)

## Set up backtesting

In [ ]:
from skforecast.model_selection import TimeSeriesFold
from sklearn.metrics import root_mean_squared_error


cv = TimeSeriesFold(
         steps=???, # =forecast horizon
         initial_train_size=???, # size of the initial training set
         refit=False, # if true, the forecaster is refitted at each fold
         fixed_train_size=False # fixed-size or expanding training data
     )

## Baseline

In [ ]:
from skforecast.recursive import ForecasterEquivalentDate
from skforecast.model_selection._validation import backtesting_forecaster


last_value_forecaster = ForecasterEquivalentDate(offset=1, n_offsets=1)

results_lv, predictions_backtest = backtesting_forecaster(
                                   forecaster=???,
                                   y=???,
                                   cv=???,
                                   metric=[root_mean_squared_error, 
                                           'mean_absolute_error'],
                                   n_jobs='auto',
                                   verbose=False,
                                   show_progress=True
                               )
results_lv.index = ["Last-Value"]
results_lv

## Decision Tree

In [ ]:
from sklearn.tree import ???
from skforecast.recursive import ForecasterRecursive
from skforecast.model_selection import grid_search_forecaster

import warnings
from skforecast.exceptions import IgnoredArgumentWarning
warnings.simplefilter('ignore', category=IgnoredArgumentWarning)


forecaster = ForecasterRecursive(
    estimator=???(random_state=7),
    lags=24
)

# Lags grid
lags_grid = [6, 12, 24]

# Hyperparameter search space
hp_grid = {
    ???
}

# list of exog features
exog_features = train_set.columns.to_list()
exog_features.remove("users")

cv_hp_search = TimeSeriesFold(
    steps=???, # =forecast horizon
    initial_train_size=???, # 80% training, 20% validation
    refit=False, # if true, the forecaster is refitted at each fold
    fixed_train_size=False # fixed-size or expanding training data
)

results_search = grid_search_forecaster(
    forecaster=forecaster,
    y=???,
    exog=???,  
    cv=???,
    param_grid=???,
    lags_grid=???,
    metric='mean_absolute_error',
    return_best=True
)

In [ ]:
results_search.head(3)

In [ ]:
best_params = results_search['params'].iat[0]
best_params

In [ ]:
best_lags = results_search['lags'].iat[0]
best_lags

## Evaluate on test

In [ ]:
results_dt, predictions = backtesting_forecaster(
                        forecaster=???, # use the forecaster with the best HPs found in the search
                        y=???,
                        exog=???,
                        cv=???,
                        metric=[root_mean_squared_error, 
                                 'mean_absolute_error'],
                     )
results_dt.index = ["Decision Tree"]
results_dt

Answers:


- At the start of HP tuning, how many observations were used for model training and how many for validation? How many were used for model evaluation on the test set?

A: ???

- What are the HPs of the best DT model? What is the optimal number of lags for this model?

A: ???

- Does the HP-tuned DT model improve on the last-value baseline? 

A: ???

- How does it compare to the Random Forest model?

A: ???

# Ex 7.3.

The dataframe below contains a time series. Extract from the index a feature to represent the trend and cyclical features to represent the daily seasonality.

In [ ]:
# create per-minute timestamps
start = "2026-01-01 00:00:00"
end = "2026-03-31 23:59:00"
index = pd.date_range(start=start, end=end, freq="min")

# generate white noise values
rng = np.random.default_rng(seed=42)
values = rng.standard_normal(len(index))

df = pd.DataFrame({"date_time": index, "value": values})
df = df.set_index("date_time")
df.head()

In [ ]:
# create a trend feature

???

In [ ]:
# create cyclical features

???

# Ex. 7.4.

Using the best HP settings for the DT model, perform feature selection, retrain a DT model on the reduced feature set and compare its performance to the best model in the previous exercise.

In [ ]:
from skforecast.feature_selection import select_features
from sklearn.feature_selection import RFECV

dt = ???(**best_params, random_state=7)
selector = RFECV(estimator=dt, step=1, cv=3)

selected_lags, selected_window_features, selected_exog = select_features(
    forecaster=forecaster,
    selector=selector, # or, SelectKBest(f_regression, k=5)
    y=???,  
    exog=???,
    subsample=1.0, 
    random_state=7,
    verbose=True
)

In [ ]:
forecaster = ForecasterRecursive(
    estimator=???(**best_params, random_state=7),
    lags=selected_lags
)

results_dt_fs, predictions = backtesting_forecaster(
                        forecaster=forecaster, 
                        y=???,
                        exog=???,
                        cv=???,
                        metric=[root_mean_squared_error, 
                                 'mean_absolute_error'],
                     )
results_dt_fs.index = ["Decision Tree (feature selection)"]
results_dt_fs

In [ ]:
tdf = pd.concat([results_lv, results_dt, results_dt_fs], axis=0)
tdf

# Ex. 7.5.

Extract feature importances from the DT model trained on the full set of features. Comments on your findings.

In [ ]:
forecaster = ForecasterRecursive(
    estimator=???(**best_params, random_state=7),
    lags=best_lags
)

???

# Citing this notebook
If you use this notebook in your work, please cite it as follows:

Pekar, V. (2026). Business Forecasting. Lecture examples and exercises. (Version 1.0.0). URL: https://github.com/vpekar/bf